# TrainLM T1: TorchTPU single-device probe

This notebook detects whether the current Python runtime can allocate a native PyTorch TPU tensor, then runs the probe on that device or explicitly on CPU. It checks package provenance, forward execution, autograd, an AdamW update, and optionally torch.compile.

Hosted Kaggle currently documents TPU v5e-8, not v5e-1. Colab's public TPU notebook documents TPU v2. Neither hosted accelerator is v5e-1 evidence. For the planned v5e-1 check, use a one-chip Cloud TPU v5e VM (Google lists machine type ct5lp-hightpu-1t) with the approved TorchTPU runtime; a Colab frontend connected to that VM runs on the VM's hardware.

This notebook does not install packages or treat a generic accelerator as TorchTPU. A TPU pass is labeled with the runtime platform and must be paired with the VM/accelerator identifier to establish the hardware generation. Use v5e-8 for checks that require multiple devices.

In [ ]:
from pathlib import Path
import subprocess

PROBE_RELATIVE_PATH = Path("examples/torchtpu_t1_v5e1_probe.py")
repo_candidates = [
    Path.cwd(), *Path.cwd().parents,
    Path("/kaggle/working/TrainLM"), Path("/content/TrainLM"),
]
repo_root = next(
    (candidate.resolve() for candidate in repo_candidates
     if (candidate / PROBE_RELATIVE_PATH).is_file()),
    None,
)

if repo_root is None:
    if Path("/kaggle/working").exists():
        clone_path = Path("/kaggle/working/TrainLM")
    elif Path("/content").exists():
        clone_path = Path("/content/TrainLM")
    else:
        clone_path = Path.cwd() / "TrainLM"
    if clone_path.exists():
        raise FileNotFoundError(
            f"{clone_path} exists but does not contain {PROBE_RELATIVE_PATH}; "
            "attach the TrainLM repository or choose an empty clone path."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Dhiraj309/TrainLM.git", str(clone_path)],
        check=True,
    )
    repo_root = clone_path.resolve()

assert (repo_root / PROBE_RELATIVE_PATH).is_file()
print(f"repository={repo_root}")

## Detect runtime and device

The official TorchTPU API uses torch.device("tpu"). The cell below tests that a small tensor can actually be allocated, so a parsed device alias alone cannot count as TPU availability. It prints the detected notebook host and device result. The standard hosted options are documented as Colab TPU v2 and Kaggle v5e-8. A Colab connection to a Cloud TPU VM uses that VM's hardware.

In [ ]:
import os
import sys

if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    NOTEBOOK_HOST = "Colab"
elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ:
    NOTEBOOK_HOST = "Kaggle"
else:
    NOTEBOOK_HOST = "other/local runtime"

import torch

DEVICE = "cpu"
TPU_DETECTION = "unavailable"
try:
    _tpu_probe = torch.empty(1, device=torch.device("tpu"))
    _tpu_probe.cpu()  # Confirm a real allocation and device-to-host read.
    DEVICE = "tpu"
    TPU_DETECTION = "native PyTorch TPU tensor allocation passed"
except Exception as exc:
    TPU_DETECTION = f"unavailable ({type(exc).__name__}: {str(exc)[:240]})"

print(f"notebook_host={NOTEBOOK_HOST}")
print(f"selected_device={DEVICE}")
print(f"tpu_detection={TPU_DETECTION}")
if NOTEBOOK_HOST == "Kaggle":
    print("hosted_accelerator_note=Kaggle documents TPU v5e-8; this is not v5e-1 evidence")
elif NOTEBOOK_HOST == "Colab":
    print("hosted_accelerator_note=standard Colab TPU is documented as TPU v2; a connected local runtime uses its server hardware")
if DEVICE == "cpu":
    print("scope=CPU-only smoke; no TPU evidence")
else:
    print("scope=single-device TPU smoke; record the VM/accelerator identifier before attributing a TPU generation")

## Eager single-device smoke

The probe uses a tiny synthetic model and data. It prints Python/PyTorch and installed Torch/TPU/XLA-related distribution versions, then checks finite loss and gradients plus an actual optimizer update. Preserve the complete cell output as evidence.

In [ ]:
import sys
import subprocess

probe = repo_root / PROBE_RELATIVE_PATH
subprocess.run(
    [sys.executable, str(probe), "--device", DEVICE] + (["--allow-cpu"] if DEVICE == "cpu" else []),
    cwd=repo_root,
    check=True,
)

## Optional compile smoke

Run this separately because compilation can take longer and may expose a runtime/compiler issue. If it fails, keep the error as evidence; the script does not silently fall back to eager execution.

In [ ]:
RUN_COMPILE = False  # Set True to run the optional torch.compile check.
if RUN_COMPILE:
    subprocess.run(
        [sys.executable, str(probe), "--device", DEVICE, "--compile"] + (["--allow-cpu"] if DEVICE == "cpu" else []),
        cwd=repo_root,
        check=True,
    )
else:
    print("compile probe skipped; set RUN_COMPILE = True when ready")

## Record results

Record the notebook host, runtime/VM image identifier, date, actual accelerator, selected device, command, Python/PyTorch/distribution versions, and whether import, device placement, forward, autograd, optimizer update, and optional compile passed. The host label identifies the notebook frontend only; keep the accelerator identifier to establish v5e-1 evidence. Do not paste tokens, credentials, or a full environment dump.

This is only a single-device smoke. Move to v5e-8 for collectives, DP8, sharding/FSDP, distributed-checkpoint behavior, or scaling; do not infer those results from this notebook.